# Stage 5–6: POI floors and vertical land-use indices
**Inputs** (from the two earlier notebooks): `poi_buildings_<AREA>.csv`, `buildings_floors_<AREA>.csv`, and the ward shapefile.

**What it does**
1. Predicts a floor band for every POI whose floor is unknown, from land use, building height/floors, location and how many POIs share the building.
2. Draws those floors at random from the predicted probabilities many times (Monte Carlo), so uncertainty is carried into every index.
3. Calculates, for **buildings with ≥ 3 POIs**, **500 m grid cells** and **wards**:

| Index | Meaning |
|---|---|
| `H_use` | Shannon entropy of land uses (the usual 2D mix), 0–1 after dividing by ln(K) |
| `H_band` | how spread the POIs are across floor bands, 0–1 |
| `MI` | mutual information between floor band and land use = `H_use − H_use|band` |
| **`VSI`** | **vertical stratification index = MI ÷ H_use**: share of the land-use mix explained by floor (0 = uses mixed on the same floors, 1 = each floor has its own use) |
| `VSI_null` | VSI expected if floors were shuffled at random within the unit (small samples inflate MI) |
| **`VSI_excess`** | `VSI − VSI_null`: stratification beyond chance; use this for comparisons |
| `VSI_conf` | share of simulations where VSI > VSI_null |

4. Joins ward population and building statistics, and tests how the indices relate to density and building height.

Change only **Cell 1 (Settings)**.

## 1. Settings

In [ ]:
import os, warnings
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import box
from IPython.display import display
warnings.filterwarnings('ignore', category=RuntimeWarning)

OUT_DIR   = r'outputs'
AREA_NAME = 'bengaluru'
POI_CSV   = os.path.join(OUT_DIR, f'poi_buildings_{AREA_NAME}.csv')
BLDG_CSV  = os.path.join(OUT_DIR, f'buildings_floors_{AREA_NAME}.csv')
SHP       = r'path/to/study_area.shp'
WARD_COL  = 'name_en'
POP_COL   = 'population'

N_SIM            = 200     # Monte Carlo draws of unknown floors
MIN_POI_BUILDING = 3       # buildings need at least this many POIs for an index
MIN_POI_AREA     = 10      # grid cells / wards need at least this many POIs
GRID_M           = 500
SEED             = 42

# Floor bands (floor numbers: ground = 0, first = 1, basement = -1)
BANDS      = ['Basement', 'Ground', '1st', '2nd–3rd', '4th–7th', '8th+']
BAND_START = np.array([-99, 0, 1, 2, 4, 8])            # lowest floor number in each band
def band_of(floor):
    f = np.asarray(floor, dtype=float)
    b = np.searchsorted(BAND_START, f, side='right') - 1
    return np.where(np.isnan(f), -1, b)

LANDUSE = ['Retail', 'Food & drink', 'Office', 'Personal services', 'Health', 'Finance',
           'Education', 'Hospitality', 'Leisure', 'Civic', 'Religious']
GROUPS = {'Retail & food': ['Retail', 'Food & drink'], 'Office': ['Office'],
          'Services': ['Personal services', 'Health', 'Finance', 'Education', 'Civic', 'Religious'],
          'Leisure & hotels': ['Leisure', 'Hospitality']}

FIG_DIR = os.path.join(OUT_DIR, f'figures_{AREA_NAME}')
os.makedirs(FIG_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)

### Chart style

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch

INK, INK2, MUTED, GRID, AXIS, SURFACE = '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7', '#fcfcfb'
BLUE, LIGHT_BLUE, NODATA = '#2a78d6', '#86b6ef', '#c3c2b7'
SEQ  = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#256abf', '#184f95', '#0d366b']
CAT  = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']          # fixed order: groups 1–4
CAT_TEXT = ['white', 'white', 'white', INK]
DIV  = ['#d03b3b', '#e98a86', '#f0efec', '#86b6ef', '#2a78d6']   # diverging, grey midpoint

mpl.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Segoe UI', 'Arial', 'DejaVu Sans'],
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'text.color': INK, 'axes.labelcolor': INK2, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': AXIS, 'axes.linewidth': 0.8, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'savefig.dpi': 200, 'savefig.bbox': 'tight'})

def save(fig, name):
    base = os.path.join(FIG_DIR, os.path.splitext(name)[0])
    fig.savefig(base + '.png'); fig.savefig(base + '.svg')
    print('saved', base + '.png / .svg')

def subtitle(ax, text):
    ax.text(0, 1.02, text, transform=ax.transAxes, fontsize=8.5, color=INK2, va='bottom')

## 2. Load POIs, buildings and wards

In [ ]:
bld = pd.read_csv(BLDG_CSV, usecols=['bid', 'name', 'floors_final', 'h_median', 'footprint_m2', 'lat', 'lon', 'floor_source'])
poi = pd.read_csv(POI_CSV)
poi = poi.drop(columns=[c for c in ['floors_final', 'h_median', 'bld_floor_source'] if c in poi.columns])
poi = poi.merge(bld[['bid', 'floors_final', 'h_median', 'footprint_m2']], on='bid', how='left')   # latest floor model
poi = poi[poi['landuse'].isin(LANDUSE)].reset_index(drop=True)
poi = gpd.GeoDataFrame(poi, geometry=gpd.points_from_xy(poi['lon'], poi['lat']), crs=4326)

wards = gpd.read_file(SHP).to_crs(4326)
UTM = wards.estimate_utm_crs()

# A POI cannot be above the building's top floor (top floor number = floors - 1)
too_high = poi['floor'].notna() & poi['floors_final'].notna() & (poi['floor'] > poi['floors_final'] - 1)
poi.loc[too_high, 'floor'] = np.nan
poi['band_obs'] = band_of(poi['floor'])                      # -1 = unknown
poi['n_poi_bld'] = poi.groupby('bid')['bid'].transform('size')
poi['has_brand'] = poi['brand'].notna().astype(int) if 'brand' in poi.columns else 0

print(f'POIs: {len(poi):,} | floor known: {(poi.band_obs >= 0).sum():,} ({(poi.band_obs >= 0).mean():.1%})'
      f' | floors removed as above building top: {too_high.sum():,}')
print('Known floor bands:', dict(zip(BANDS, np.bincount(poi.loc[poi.band_obs >= 0, 'band_obs'], minlength=6))))

## 3. Predict the floor band of POIs with unknown floor
Gradient-boosted classifier trained on POIs with a known floor. Predictions are then restricted to bands the building physically has.
**Caution:** POIs with known floors are concentrated in malls and tech parks, so the model learns most from those settings.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold

lu_code = {u: i for i, u in enumerate(LANDUSE)}
def features(df):
    return np.column_stack([df['landuse'].map(lu_code).values, df['floors_final'].values, df['h_median'].values,
                            np.log1p(df['footprint_m2'].values), df['n_poi_bld'].fillna(1).values,
                            df['has_brand'].values, df['lon'].values, df['lat'].values]).astype(float)

def feasible(df):
    # bands a building has: band start floor <= top floor (floors - 1). Basement always allowed.
    top = (df['floors_final'] - 1).values
    m = np.ones((len(df), len(BANDS)), bool)
    known = ~np.isnan(top)
    m[known, 1:] = BAND_START[1:][None, :] <= top[known, None]
    m[:, 1] = True                                              # ground always exists
    return m

def make_model():
    return HistGradientBoostingClassifier(categorical_features=[0], max_iter=250, learning_rate=0.06,
                                          max_leaf_nodes=15, l2_regularization=1.0, random_state=SEED)

def proba_full(model, X, mask):
    p = np.zeros((len(X), len(BANDS)))
    p[:, model.classes_] = model.predict_proba(X)
    p *= mask
    s = p.sum(1, keepdims=True)
    p = np.where(s > 0, p / np.where(s == 0, 1, s), np.eye(len(BANDS))[1])   # nothing feasible -> ground
    return p

train = poi[poi['band_obs'] >= 0]
X, y, M = features(train), train['band_obs'].values, feasible(train)

# cross-validated check against a simple baseline (most common band for the POI's land use)
k = int(min(5, np.bincount(y)[np.bincount(y) > 0].min()))
cv_rows, oof = [], np.zeros((len(train), len(BANDS)))
for tr, te in StratifiedKFold(max(k, 2), shuffle=True, random_state=SEED).split(X, y):
    m = make_model().fit(X[tr], y[tr])
    p = proba_full(m, X[te], M[te]); oof[te] = p
    base = train.iloc[tr].groupby('landuse')['band_obs'].agg(lambda s: s.mode()[0])
    yb = train.iloc[te]['landuse'].map(base).fillna(1).values
    cv_rows.append({'model_accuracy': (p.argmax(1) == y[te]).mean(), 'baseline_accuracy': (yb == y[te]).mean(),
                    'log_loss': -np.mean(np.log(np.clip(p[np.arange(len(te)), y[te]], 1e-9, 1)))})
cv_imp = pd.DataFrame(cv_rows).mean()
display(cv_imp.round(3).to_frame('5-fold mean'))

model = make_model().fit(X, y)
P = proba_full(model, features(poi), feasible(poi))
known = poi['band_obs'].values >= 0
P[known] = np.eye(len(BANDS))[poi.loc[known, 'band_obs'].values]     # known floors are fixed
poi[[f'p_{b}' for b in BANDS]] = P
print('Expected POIs per band (known + predicted):', dict(zip(BANDS, P.sum(0).round(0))))

## 4. Monte Carlo indices for buildings, grid cells and wards
Each draw assigns every unknown POI a floor band from its probabilities, then computes the indices. A second, shuffled copy (floors randomly permuted within each unit) gives the chance level `VSI_null`.

In [ ]:
K, B = len(LANDUSE), len(BANDS)
use = poi['landuse'].map(lu_code).values
cumP = P.cumsum(1)

def draw_bands():
    u = rng.random(len(P))[:, None]
    return np.minimum((u > cumP).sum(1), B - 1)

def H(p, axis):
    return -(np.where(p > 0, p * np.log(p), 0)).sum(axis)

def indices(unit, n_units, band, use_):
    c = np.bincount(unit * B * K + band * K + use_, minlength=n_units * B * K).reshape(n_units, B, K).astype(float)
    n = c.sum((1, 2)); nn = np.where(n == 0, 1, n)
    h_use  = H(c.sum(1) / nn[:, None], 1)
    h_band = H(c.sum(2) / nn[:, None], 1)
    h_join = H((c / nn[:, None, None]).reshape(n_units, -1), 1)
    mi = h_use + h_band - h_join
    vsi = np.where(h_use > 1e-9, mi / np.where(h_use > 1e-9, h_use, 1), np.nan)
    return h_use, h_band, h_join, mi, vsi

def shuffle_within(unit, band):
    order = np.lexsort((rng.random(len(unit)), unit))      # random order inside each unit
    base = np.argsort(unit, kind='stable')
    out = np.empty_like(band); out[base] = band[order]
    return out

# ---- define the three sets of units ----
pts_utm = poi.to_crs(UTM)
poi['cell'] = ((pts_utm.geometry.x // GRID_M).astype(int).astype(str) + '_' +
               (pts_utm.geometry.y // GRID_M).astype(int).astype(str))
SCALES = {}
for scale, col, mn in [('building', 'bid', MIN_POI_BUILDING), ('grid', 'cell', MIN_POI_AREA), ('ward', 'ward', MIN_POI_AREA)]:
    if col not in poi.columns: continue
    cnt = poi[col].value_counts()
    keep_ids = cnt[cnt >= mn].index
    sel = poi[col].isin(keep_ids).values
    ids, unit = np.unique(poi.loc[sel, col].values, return_inverse=True)
    SCALES[scale] = dict(sel=sel, ids=ids, unit=unit, n=len(ids))
    print(f'{scale:8s}: {len(ids):,} units with ≥ {mn} POIs, covering {sel.sum():,} POIs')

store = {s: {k: np.zeros((N_SIM, d['n'])) for k in ['H_band', 'MI', 'VSI', 'VSI_null', 'H_joint']} for s, d in SCALES.items()}
static = {}
band_counts = np.zeros((N_SIM, B, K))
for sim in range(N_SIM):
    band = draw_bands()
    band_counts[sim] = np.bincount(band * K + use, minlength=B * K).reshape(B, K)
    for s, d in SCALES.items():
        b_, u_ = band[d['sel']], use[d['sel']]
        hu, hb, hj, mi, vsi = indices(d['unit'], d['n'], b_, u_)
        _, _, _, _, vsi0 = indices(d['unit'], d['n'], shuffle_within(d['unit'], b_), u_)
        st = store[s]; st['H_band'][sim], st['MI'][sim], st['VSI'][sim], st['VSI_null'][sim], st['H_joint'][sim] = hb, mi, vsi, vsi0, hj
        static[s] = hu
    if (sim + 1) % 50 == 0: print(f'{sim + 1}/{N_SIM} draws')

def summarise(s):
    st, d = store[s], SCALES[s]
    out = pd.DataFrame({'id': d['ids'], 'n_poi': np.bincount(d['unit'], minlength=d['n']),
                        'H_use': static[s] / np.log(K),
                        'H_band': np.nanmean(st['H_band'], 0) / np.log(B),
                        'H_joint': np.nanmean(st['H_joint'], 0) / np.log(B * K),
                        'MI': np.nanmean(st['MI'], 0),
                        'VSI': np.nanmean(st['VSI'], 0),
                        'VSI_p05': np.nanpercentile(st['VSI'], 5, 0), 'VSI_p95': np.nanpercentile(st['VSI'], 95, 0),
                        'VSI_null': np.nanmean(st['VSI_null'], 0)})
    out['VSI_excess'] = out['VSI'] - out['VSI_null']
    out['VSI_conf'] = np.nanmean(st['VSI'] > st['VSI_null'], 0)
    return out

idx = {s: summarise(s) for s in SCALES}
for s, t in idx.items():
    print(f'\n{s}:'); display(t[['n_poi', 'H_use', 'H_band', 'VSI', 'VSI_null', 'VSI_excess', 'VSI_conf']].describe().round(3).loc[['mean', '50%', 'std']])

## 5. Join ward population and building statistics; save

In [ ]:
# ---- buildings ----
b_idx = idx['building'].rename(columns={'id': 'bid'}).merge(bld[['bid', 'name', 'floors_final', 'h_median', 'lat', 'lon']], on='bid', how='left')
top_use = poi.groupby('bid')['landuse'].agg(lambda s: s.value_counts().index[0])
b_idx['main_use'] = b_idx['bid'].map(top_use)
b_idx.to_csv(os.path.join(OUT_DIR, f'vertical_index_buildings_{AREA_NAME}.csv'), index=False)

# ---- grid ----
g_idx = idx['grid'].rename(columns={'id': 'cell'})
xy = g_idx['cell'].str.split('_', expand=True).astype(int) * GRID_M
grid = gpd.GeoDataFrame(g_idx, geometry=[box(x, y, x + GRID_M, y + GRID_M) for x, y in zip(xy[0], xy[1])], crs=UTM).to_crs(4326)
grid.to_file(os.path.join(OUT_DIR, f'vertical_index_grid_{AREA_NAME}.gpkg'), driver='GPKG')

# ---- wards: indices + population + building statistics ----
w = wards[[WARD_COL, POP_COL, 'geometry']].copy() if POP_COL in wards.columns else wards[[WARD_COL, 'geometry']].copy()
w['area_km2'] = w.to_crs(UTM).area / 1e6
bpts = gpd.GeoDataFrame(bld[['floors_final', 'footprint_m2']], geometry=gpd.points_from_xy(bld['lon'], bld['lat']), crs=4326)
bj = gpd.sjoin(bpts, w[[WARD_COL, 'geometry']], predicate='within', how='inner')
bj['floor_area'] = bj['footprint_m2'] * bj['floors_final'].fillna(1)
bstat = bj.groupby(WARD_COL).agg(buildings=('floors_final', 'size'), mean_floors=('floors_final', 'mean'),
                                 share_4plus=('floors_final', lambda s: (s >= 4).mean()), floor_area_m2=('floor_area', 'sum'))
w = w.merge(bstat, left_on=WARD_COL, right_index=True, how='left')
w = w.merge(idx['ward'].rename(columns={'id': WARD_COL}), on=WARD_COL, how='left')
w['far'] = w['floor_area_m2'] / (w['area_km2'] * 1e6)
w['poi_density'] = poi['ward'].value_counts().reindex(w[WARD_COL]).values / w['area_km2']   # POI 'ward' = ward name from the POI notebook
if POP_COL in w.columns:
    w['pop_density'] = pd.to_numeric(w[POP_COL], errors='coerce') / w['area_km2']
w.to_file(os.path.join(OUT_DIR, f'vertical_index_wards_{AREA_NAME}.gpkg'), driver='GPKG')
w.drop(columns='geometry').to_csv(os.path.join(OUT_DIR, f'vertical_index_wards_{AREA_NAME}.csv'), index=False)

# ---- ward-level relationships (Spearman rank correlation) ----
targets = [c for c in ['pop_density', 'poi_density', 'mean_floors', 'share_4plus', 'far'] if c in w.columns]
wv = w[w['n_poi'] >= MIN_POI_AREA]
corr = wv[['H_use', 'H_band', 'VSI', 'VSI_excess'] + targets].corr(method='spearman').loc[['H_use', 'H_band', 'VSI', 'VSI_excess'], targets]
print(f'Spearman correlations across {len(wv)} wards with ≥ {MIN_POI_AREA} POIs:'); display(corr.round(2))
print('Saved building, grid and ward index files to', OUT_DIR)

---
# Figures

### E1. Floor prediction: accuracy and predicted vs observed bands

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [1, 1.6]})
vals = [cv_imp['baseline_accuracy'], cv_imp['model_accuracy']]
bars = a1.barh(['Most common band\nfor the land use', 'Gradient-boosted\nmodel'], vals, color=[LIGHT_BLUE, BLUE], height=0.5)
a1.bar_label(bars, labels=[f'{v:.0%}' for v in vals], padding=4, color=INK2)
a1.set_xlim(0, 1); a1.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1)); a1.grid(axis='y', visible=False)
a1.set_title('Exact floor band correct', pad=20); subtitle(a1, f'5-fold cross-validation on {len(train):,} POIs with known floor')

obs = np.bincount(y, minlength=B) / len(y); pred = oof.mean(0)
x = np.arange(B); wdt = 0.38
a2.bar(x - wdt / 2, obs, wdt, color=BLUE, label='Observed')
a2.bar(x + wdt / 2, pred, wdt, color=LIGHT_BLUE, label='Predicted (held out)')
a2.set_xticks(x, BANDS); a2.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1)); a2.grid(axis='x', visible=False)
a2.set_title('Share of POIs by floor band', pad=20); subtitle(a2, 'Predicted shares should match observed ones')
a2.legend(frameon=False)
fig.tight_layout(); save(fig, 'e1_floor_prediction.png'); plt.show()

### E2. City-wide vertical profile: which uses sit on which floors

In [ ]:
mean_counts = band_counts.mean(0)                                  # B x K, averaged over draws
gcounts = np.column_stack([mean_counts[:, [lu_code[u] for u in us]].sum(1) for us in GROUPS.values()])
share = gcounts / gcounts.sum(1, keepdims=True)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={'width_ratios': [2.2, 1]}, sharey=True)
left = np.zeros(B)
for j, (gname, col, tcol) in enumerate(zip(GROUPS, CAT, CAT_TEXT)):
    a1.barh(BANDS, share[:, j], left=left, color=col, edgecolor=SURFACE, linewidth=2, height=0.62, label=gname)
    for i in range(B):
        if share[i, j] >= 0.08:
            a1.text(left[i] + share[i, j] / 2, i, f'{share[i, j]:.0%}', ha='center', va='center', color=tcol, fontsize=8.5)
    left += share[:, j]
a1.set_xlim(0, 1); a1.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1)); a1.grid(axis='y', visible=False)
a1.set_title('Land-use mix on each floor band', pad=20); subtitle(a1, f'Mean of {N_SIM} Monte Carlo draws (known + predicted floors)')
a1.legend(frameon=False, ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.12), fontsize=9)
tot = gcounts.sum(1)
b2 = a2.barh(BANDS, tot, color=BLUE, height=0.62)
a2.bar_label(b2, labels=[f'{v:,.0f}' for v in tot], padding=3, fontsize=8.5, color=INK2)
a2.set_title('POIs per band', pad=20); a2.grid(axis='y', visible=False); a2.margins(x=0.25)
a2.xaxis.set_major_formatter(mpl.ticker.StrMethodFormatter('{x:,.0f}'))
fig.tight_layout(); save(fig, 'e2_vertical_profile.png'); plt.show()

### E3. Buildings: land-use mix vs vertical stratification
Each dot is a building with ≥ 3 POIs (size = number of POIs). Bottom-left: few uses; right-bottom: many uses on the same floors; right-top: many uses separated by floor.

In [ ]:
d = b_idx[b_idx['H_use'] > 0].copy()
fig, ax = plt.subplots(figsize=(8, 6))
ax.axhline(0, color=AXIS, lw=0.8)
ax.scatter(d['H_use'], d['VSI_excess'], s=np.clip(d['n_poi'], 3, 60) * 3, color=BLUE, alpha=0.45,
           edgecolor=SURFACE, linewidth=0.5, rasterized=True)
named = d[d['name'].fillna('').str.strip() != ''].nlargest(5, 'n_poi')       # label only named buildings
named = named.assign(_k=named['VSI_excess'] - 0.3 * named['H_use']).sort_values('_k', ascending=False)   # order that keeps leader lines from crossing
for k_, (_, r) in enumerate(named.iterrows()):
    ax.annotate(f"{r['name']} ({r.n_poi} POIs)", (r['H_use'], r['VSI_excess']), xytext=(0.70, 0.93 - k_ * 0.055),
                textcoords='axes fraction', fontsize=8, color=INK2, va='center', ha='left',
                bbox=dict(boxstyle='square,pad=0.15', facecolor=SURFACE, edgecolor='none'),
                arrowprops=dict(arrowstyle='-', color=MUTED, lw=0.6, relpos=(0, 0.5), shrinkA=2, shrinkB=3))
ax.set(xlabel='Land-use mix  H_use  (0 = one use, 1 = all uses equally)',
       ylabel='Vertical stratification beyond chance  (VSI − VSI_null)')
ax.set_title('Mixed-use buildings: are uses stacked by floor?', pad=20)
subtitle(ax, f'{len(d):,} buildings with ≥ {MIN_POI_BUILDING} POIs and more than one land use  ·  '
             f'{(d.VSI_conf >= 0.9).mean():.0%} stratified in ≥ 90% of draws')
for s_, lab in [(5, '5 POIs'), (20, '20'), (60, '60+')]:
    ax.scatter([], [], s=s_ * 3, color=BLUE, alpha=0.45, label=lab)
ax.legend(frameon=False, title='POIs in building', fontsize=8.5, title_fontsize=8.5, loc='upper left', labelspacing=1.2)
fig.tight_layout(); save(fig, 'e3_buildings_mix_vs_stratification.png'); plt.show()

### E4. Ward maps: horizontal mix, vertical spread and vertical stratification

In [ ]:
cmap = mpl.colors.LinearSegmentedColormap.from_list('seq', SEQ)
dmap = mpl.colors.LinearSegmentedColormap.from_list('div', DIV)
panels = [('H_use', 'Land-use mix (2D entropy)', cmap, None),
          ('H_band', 'Spread across floor bands', cmap, None),
          ('VSI_excess', 'Vertical stratification beyond chance', dmap, 'div')]
fig, axes = plt.subplots(1, 3, figsize=(17, 6.5))
for ax, (col, title, cm, kind) in zip(axes, panels):
    v = w[col].where(w['n_poi'] >= MIN_POI_AREA)
    if kind == 'div':
        lim = np.nanpercentile(np.abs(v), 97); vmin, vmax = -lim, lim
    else:
        vmin, vmax = 0, np.nanpercentile(v, 98)
    w.assign(_v=v).plot(column='_v', ax=ax, cmap=cm, vmin=vmin, vmax=vmax, edgecolor=SURFACE, linewidth=0.3,
                        missing_kwds={'color': NODATA, 'edgecolor': SURFACE, 'linewidth': 0.3})
    ax.set_axis_off(); ax.set_title(title)
    cb = fig.colorbar(mpl.cm.ScalarMappable(cmap=cm, norm=mpl.colors.Normalize(vmin, vmax)), ax=ax, shrink=0.55, pad=0.01)
    cb.outline.set_visible(False); cb.ax.tick_params(labelsize=8)
fig.text(0.01, 0.02, f'Grey: fewer than {MIN_POI_AREA} POIs. Right map: blue = uses separated by floor more than chance, red = less than chance.',
         color=MUTED, fontsize=9)
fig.suptitle(f'{AREA_NAME.title()}: vertical land-use indices by ward', x=0.01, ha='left', fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0.04, 1, 0.95]); save(fig, 'e4_ward_maps.png'); plt.show()

### E5. Grid map: vertical stratification beyond chance

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
gp = grid.to_crs(UTM)
lim = np.nanpercentile(np.abs(gp['VSI_excess']), 97)
gp.plot(column='VSI_excess', ax=ax, cmap=dmap, vmin=-lim, vmax=lim, edgecolor=SURFACE, linewidth=0.3,
        missing_kwds={'color': NODATA})
w.to_crs(UTM).boundary.plot(ax=ax, color=AXIS, linewidth=0.4, zorder=3)                  # ward boundaries (light)
outline = w.to_crs(UTM).union_all().buffer(30).buffer(-30)          # closes tiny gaps between wards
gpd.GeoSeries([outline], crs=UTM).boundary.plot(ax=ax, color=INK2, linewidth=1.2, zorder=4)   # city outline
ax.set_axis_off()
fig.suptitle(f'Vertical stratification beyond chance, {GRID_M} m cells', x=0.02, ha='left', fontsize=12, fontweight='bold')
fig.text(0.02, 0.935, f'{len(gp):,} cells with ≥ {MIN_POI_AREA} POIs  ·  blue = uses separated by floor, red = mixed on the same floors'
         '  ·  grey = one land use only', fontsize=8.5, color=INK2)
cb = fig.colorbar(mpl.cm.ScalarMappable(cmap=dmap, norm=mpl.colors.Normalize(-lim, lim)), ax=ax, shrink=0.5, pad=0.01)
cb.outline.set_visible(False)
fig.tight_layout(rect=[0, 0, 1, 0.93]); save(fig, 'e5_grid_vsi.png'); plt.show()

### E6. Do vertical patterns follow density and height? (wards)

In [ ]:
xs = [(c, l) for c, l in [('pop_density', 'Population per km²'), ('poi_density', 'POIs per km²'),
                          ('mean_floors', 'Mean floors per building'), ('far', 'Built floor area ÷ land area')] if c in wv.columns]
fig, axes = plt.subplots(1, len(xs), figsize=(4.2 * len(xs), 4), sharey=True)
for ax, (c, lab) in zip(np.atleast_1d(axes), xs):
    ax.scatter(wv[c], wv['VSI_excess'], s=18, color=BLUE, alpha=0.7, edgecolor=SURFACE, linewidth=0.5)
    if c in ('pop_density', 'poi_density'): ax.set_xscale('log')
    rho = wv[[c, 'VSI_excess']].corr(method='spearman').iloc[0, 1]
    ax.set_title(f'Spearman ρ = {rho:+.2f}', fontsize=10)
    ax.set_xlabel(lab); ax.axhline(0, color=AXIS, lw=0.8)
np.atleast_1d(axes)[0].set_ylabel('VSI − VSI_null')
fig.suptitle(f'Ward vertical stratification vs density and height ({len(wv)} wards)', x=0.01, ha='left', fontsize=12, fontweight='bold')
fig.tight_layout(); save(fig, 'e6_ward_relationships.png'); plt.show()

### E7. Change 2016 → 2023 vs today's vertical stratification (optional)
Needs `building_change_<AREA>.csv` from section 11 of the heights notebook. POIs are only available for today, so this shows an association between growth and current stratification, not change in stratification.

In [ ]:
CHG_CSV = os.path.join(OUT_DIR, f'building_change_{AREA_NAME}.csv')
assert os.path.exists(CHG_CSV), 'Run section 11 of the heights notebook first (building_change file not found)'
EARLY, LATE = [2016, 2017], [2022, 2023]
chg = pd.read_csv(CHG_CSV)
cg = gpd.GeoDataFrame(chg, geometry=gpd.points_from_xy(chg['lon'], chg['lat']), crs=4326)

# ---- ward level ----
cj = gpd.sjoin(cg, w[[WARD_COL, 'geometry']], predicate='within', how='inner')
wc = cj.groupby(WARD_COL).agg(
        share_new=('change', lambda s: (s == 'new').mean()),
        share_taller=('change', lambda s: (s == 'taller').mean()),
        added_floor_area_m2=('added_floor_area_m2', 'sum'))
w2 = w.drop(columns=[c for c in wc.columns if c in w.columns]).merge(wc, left_on=WARD_COL, right_index=True, how='left')
w2['added_far'] = w2['added_floor_area_m2'] / (w2['area_km2'] * 1e6)      # new floor area ÷ land area, 2016→2023
w2.drop(columns='geometry').to_csv(os.path.join(OUT_DIR, f'ward_change_vs_vertical_{AREA_NAME}.csv'), index=False)

wv2 = w2[w2['n_poi'] >= MIN_POI_AREA]
change_cols = ['share_new', 'share_taller', 'added_far']
print(f'Spearman correlation, {len(wv2)} wards: 2016→2023 change vs today\'s vertical indices')
display(wv2[change_cols + ['H_use', 'H_band', 'VSI_excess']].corr(method='spearman')
        .loc[change_cols, ['H_use', 'H_band', 'VSI_excess']].round(2))

# ---- grid level (same cells as the vertical index) ----
cu = cg.to_crs(UTM)
cg['cell'] = (cu.geometry.x // GRID_M).astype(int).astype(str) + '_' + (cu.geometry.y // GRID_M).astype(int).astype(str)
gc = cg.groupby('cell').agg(added_floor_area_m2=('added_floor_area_m2', 'sum'),
                            share_taller=('change', lambda s: (s == 'taller').mean()))
grid2 = grid.merge(gc, on='cell', how='left')
grid2['added_far'] = grid2['added_floor_area_m2'] / GRID_M ** 2

# ---- Figure E7: where the city grew, and does it match vertical stratification? ----
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 7), gridspec_kw={'width_ratios': [1.15, 1]})
cells = gpd.GeoDataFrame(gc.reset_index(), geometry=[box(int(x) * GRID_M, int(y) * GRID_M, (int(x) + 1) * GRID_M, (int(y) + 1) * GRID_M)
                          for x, y in gc.index.str.split('_')], crs=UTM)
cells['added_far'] = cells['added_floor_area_m2'] / GRID_M ** 2
vmax = np.nanpercentile(cells['added_far'], 98)
cmap = mpl.colors.LinearSegmentedColormap.from_list('seq', SEQ)
cells.plot(column='added_far', ax=a1, cmap=cmap, vmin=0, vmax=vmax, edgecolor=SURFACE, linewidth=0.2)
w.to_crs(UTM).boundary.plot(ax=a1, color=AXIS, linewidth=0.4)
gpd.GeoSeries([w.to_crs(UTM).union_all().buffer(30).buffer(-30)], crs=UTM).boundary.plot(ax=a1, color=INK2, linewidth=1.2)
a1.set_axis_off(); a1.set_title(f'New floor area added {EARLY[0]}–{LATE[-1]}', pad=20)
subtitle(a1, f'Per {GRID_M} m cell, as a share of land area (new buildings + added floors)')
cb = fig.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=mpl.colors.Normalize(0, vmax)), ax=a1, shrink=0.5, pad=0.01)
cb.outline.set_visible(False)

a2.scatter(wv2['added_far'], wv2['VSI_excess'], s=22, color=BLUE, alpha=0.75, edgecolor=SURFACE, linewidth=0.5)
a2.axhline(0, color=AXIS, lw=0.8)
rho = wv2[['added_far', 'VSI_excess']].corr(method='spearman').iloc[0, 1]
a2.set(xlabel=f'New floor area ÷ ward land area, {EARLY[0]}–{LATE[-1]}', ylabel='Vertical stratification beyond chance (today)')
a2.set_title('Are the wards that grew most more stratified today?', pad=20)
subtitle(a2, f'{len(wv2)} wards  ·  Spearman ρ = {rho:+.2f}')
fig.tight_layout(); save(fig, 'e7_growth_vs_stratification.png'); plt.show()

In [ ]:
# ---- OSM completeness: Open Buildings built-up area per ward (2023) ----
import ee, geemap, geopandas as gpd, pandas as pd, os
ee.Initialize(project='your-ee-project-id')

SHP     = r'path/to/study_area.shp'
OUT_DIR = r'outputs'

wards = gpd.read_file(SHP)[['name_en', 'geometry']].to_crs(4326)
fc = geemap.gdf_to_ee(wards)

ob = (ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1')
        .filterBounds(fc)
        .filterDate('2023-01-01', '2024-01-01')
        .mosaic())
pres = ob.select('building_presence')
px = ee.Image.pixelArea()

# built-up area at three presence thresholds (0.5 = main, 0.3/0.7 = sensitivity)
img = ee.Image.cat([
    pres.gt(0.3).multiply(px).rename('ob_m2_t03'),
    pres.gt(0.5).multiply(px).rename('ob_m2_t05'),
    pres.gt(0.7).multiply(px).rename('ob_m2_t07'),
])

res = img.reduceRegions(collection=fc, reducer=ee.Reducer.sum(), scale=4, tileScale=8)
df = geemap.ee_to_df(res)[['name_en', 'ob_m2_t03', 'ob_m2_t05', 'ob_m2_t07']]

out = os.path.join(OUT_DIR, 'ward_openbuildings_builtup.csv')
df.to_csv(out, index=False)
print(df.describe())
print('saved:', out)